Importing Libraries For Data-Cleaning


In [1]:
import pandas as pd
import numpy as np

# CONFIGURATION

In [2]:
INPUT_PATH  = "../data/raw/Airbnb_Open_Data.csv"

# Load raw data

In [3]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

# STEP 1 ── Drop useless / redundant columns

In [4]:
DROP_COLUMNS = ["license", "country", "country code", "house_rules", "NAME"]

df.drop(columns=DROP_COLUMNS, inplace=True)

# STEP 2 ── Rename columns to consistent snake_case
for Python handling fields

In [5]:
RENAME_MAP = {
    "host id"                       : "host_id",
    "host name"                     : "host_name",
    "neighbourhood group"           : "borough",
    "neighbourhood"                 : "neighbourhood",
    "room type"                     : "room_type",
    "Construction year"             : "construction_year",
    "service fee"                   : "service_fee",
    "minimum nights"                : "minimum_nights",
    "number of reviews"             : "number_of_reviews",
    "last review"                   : "last_review",
    "reviews per month"             : "reviews_per_month",
    "review rate number"            : "review_rate",
    "calculated host listings count": "host_listings_count",
    "availability 365"              : "availability_365",
    "host_identity_verified"        : "host_verified",
    "instant_bookable"              : "instant_bookable",
    "cancellation_policy"           : "cancellation_policy",
}

df.rename(columns=RENAME_MAP, inplace=True)

# STEP 3 ── Remove duplicate listings (by id)

In [6]:
before = len(df)
df.drop_duplicates(subset="id", keep="first", inplace=True)
removed = before - len(df)

# STEP 4 ── Clean currency columns → numeric float

Raw values look like: "$966 ", "$1,060 " — strip $, commas, whitespace

In [7]:
def clean_currency(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
              .str.replace(r"[\$,\s]", "", regex=True)
              .replace("nan", np.nan)
              .astype(float)
    )

df["price"]       = clean_currency(df["price"])
df["service_fee"] = clean_currency(df["service_fee"])

# STEP 5 ── Fix borough typos + fill nulls
 Example:- Found 'brookln' (typo for Brooklyn) and 'manhatan' (typo for Manhattan)

In [8]:
BOROUGH_TYPOS = {"brookln": "Brooklyn", "manhatan": "Manhattan"}
df["borough"] = df["borough"].replace(BOROUGH_TYPOS)
borough_mode  = df["borough"].mode()[0]
df["borough"] = df["borough"].fillna(borough_mode)

# STEP 6 ── Handle null values

In [9]:
df.isna().sum()

id                         0
host_id                    0
host_verified            289
host_name                404
borough                    0
neighbourhood             16
lat                        8
long                       8
instant_bookable         105
cancellation_policy       76
room_type                  0
construction_year        214
price                    247
service_fee              273
minimum_nights           400
number_of_reviews        183
last_review            15832
reviews_per_month      15818
review_rate              319
host_listings_count      319
availability_365         448
dtype: int64

  Column                  Nulls    Strategy

  * host_name               406      Fill → 'Unknown' (identifier, not metric)

  * host_verified           289      Fill → mode (unconfirmed/verified)

  * neighbourhood           16       Fill → borough (parent geography)

  * lat / long              8        Drop rows (cannot impute coordinates)

  * instant_bookable        105      Fill → mode

  * cancellation_policy     76       Fill → mode

  * construction_year       214      Fill → median (integer year)

  * price                   247      Drop rows (core metric, can't impute)

  * service_fee             273      Fill → 18.5% of price (Airbnb typical rate)

  * minimum_nights          409      Fill → median

  * number_of_reviews       183      Fill → 0 (new listings with no reviews)

  * last_review             15893    Keep as NaT (no review = no date; valid)

  * reviews_per_month       15879    Fill → 0 (correlates with no reviews)

  * review_rate             326      Fill → median

  * host_listings_count     319      Fill → 1 (conservative default)

  * availability_365        448      Fill → median

In [10]:
df["host_name"] = df["host_name"].fillna("Unknown")

df["host_verified"] = df["host_verified"].fillna(df["host_verified"].mode()[0])
df["neighbourhood"] = df["neighbourhood"].fillna(df["borough"])

df.dropna(subset=["lat", "long"], inplace=True)

df["instant_bookable"] = (
    df["instant_bookable"]
    .fillna(df["instant_bookable"].mode()[0])
    .astype(bool))
df["cancellation_policy"] = df["cancellation_policy"].fillna(
    df["cancellation_policy"].mode()[0])
df["construction_year"] = (
    df["construction_year"]
    .fillna(df["construction_year"].median())
    .astype(int))

df.dropna(subset=["price"], inplace=True)

svc_null_mask = df["service_fee"].isnull()

df.loc[svc_null_mask, "service_fee"] = (df.loc[svc_null_mask, "price"] * 0.185).round(2)

df["minimum_nights"] = df["minimum_nights"].fillna(df["minimum_nights"].median())

df["number_of_reviews"] = df["number_of_reviews"].fillna(0).astype(int)

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

df["review_rate"] = df["review_rate"].fillna(df["review_rate"].median()).astype(int)

df["host_listings_count"] = df["host_listings_count"].fillna(1).astype(int)

df["availability_365"] = df["availability_365"].fillna(df["availability_365"].median())

/var/folders/b7/m_nr71s115ld7m85whyltltm0000gn/T/ipykernel_58405/3128493064.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df["instant_bookable"].mode()[0])


# STEP 7 ── Fix data quality issues

In [11]:
# minimum_nights: clamp negatives to 1; cap extreme values at 365
before = df["minimum_nights"].copy()
df["minimum_nights"] = df["minimum_nights"].clip(lower=1, upper=365).astype(int)
print(f"\n[7a] minimum_nights: clamped {(before < 1).sum()} negatives → 1; "
      f"capped {(before > 365).sum()} values > 365 → 365")

# availability_365: must be in [0, 365]
df["availability_365"] = df["availability_365"].clip(lower=0, upper=365).astype(int)
print(f"[7b] availability_365: clamped to valid range [0, 365]")

# price: remove clearly erroneous prices (≤ 0)
before = len(df)
df = df[df["price"] > 0]
print(f"[7c] Removed {before - len(df)} rows with price ≤ 0")

# service_fee: clamp any negative values to 0
df["service_fee"] = df["service_fee"].clip(lower=0)


[7a] minimum_nights: clamped 13 negatives → 1; capped 35 values > 365 → 365
[7b] availability_365: clamped to valid range [0, 365]
[7c] Removed 0 rows with price ≤ 0


# STEP 8 ── Cast correct data types

In [12]:
df["last_review"]   = pd.to_datetime(df["last_review"], errors="coerce")
df["host_verified"]   = df["host_verified"].astype("category")
df["cancellation_policy"] = df["cancellation_policy"].astype("category")
df["room_type"] = df["room_type"].astype("category")
df["borough"] = df["borough"].astype("category")
df["price"]  = df["price"].astype(float)
df["service_fee"] = df["service_fee"].astype(float)
df["reviews_per_month"] = df["reviews_per_month"].astype(float)

# STEP 9 ── Add derived dashboard-ready columns

In [13]:
df["total_price"] = (df["price"] + df["service_fee"]).round(2)


df["price_tier"] = pd.cut(
    df["price"],
    bins   = [0, 75, 150, 300, 600, 10_001],
    labels = ["Budget", "Economy", "Mid-range", "Premium", "Luxury"]
)


df["high_availability"] = df["availability_365"] > 180

df["review_year"] = df["last_review"].dt.year.fillna(0).astype(int)

# STEP 10 ── Reorder columns logically

In [14]:
FINAL_COLUMNS = [
    "id", "host_id", "host_name", "host_verified", "host_listings_count",
    "borough", "neighbourhood", "lat", "long",
    "room_type", "construction_year", "instant_bookable", "cancellation_policy",
    "minimum_nights", "availability_365", "high_availability",
    "price", "service_fee", "total_price", "price_tier",
    "number_of_reviews", "last_review", "review_year", "reviews_per_month", "review_rate",
]

df = df[FINAL_COLUMNS]

# STEP 11 -- Set output file path

In [ ]:
OUTPUT_PATH = "Airbnb_Cleaned.csv"

# STEP 12 -- Save cleaned dataset

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)